# CRNT diagnostic: tool or agent, and where the time goes

Runs the agent's own CRNT tool (`crnt_bistability`) step by step on one model:

1. **Structure only**: network size and deficiency (fast).
2. **Stage timing**: the same crnt4sbml calls the tool makes, timed one by one, to find which stage is slow.
3. **The tool itself**: exactly what the agent calls, with a timer.
4. **The agent** (optional): the same question through the full agent.

If step 3 is slow, the problem is the tool (crnt4sbml on this network). If step 3 is fast but step 4 is slow, the problem is the agent.

In [ ]:
import os, sys, time, json
PROJECT = os.path.abspath("..")            # this notebook lives in notebooks/
sys.path.insert(0, PROJECT)
from dynalyst import config

MODEL = ""                                 # the model to check (name, file or path)
ITERATIONS = 10                            # optimisation starts; more = more thorough, slower
config.CRNT_MAX_SECONDS = 3600             # the tool's own limit is 600 s; raised for this diagnosis

## 1. Structure only (network size, deficiency)

In [ ]:
from dynalyst.analysis.crnt_analysis import crnt_bistability

t = time.time()
s = json.loads(crnt_bistability(MODEL, structure_only=True))
print(f"structure only: {time.time() - t:.1f} s")
print("result fields:", list(s))
print("conclusion:", s.get("conclusion"))
CRNT_SBML = os.path.join(PROJECT, s["crnt_sbml"])      # the mass-action network the tool built
print("network file:", CRNT_SBML)

## 2. Stage timing (tool internals)

The same crnt4sbml steps the tool runs, with our compatibility fixes and speed-ups (`import_crnt4sbml`). The slowest stage tells you what to change:

- **mass-conservation setup** slow: network size (symbolic algebra). Iterations do not help.
- **optimisation** slow: scales with `ITERATIONS`; lower it, or accept the wait.

In [ ]:
from dynalyst.analysis.crnt_network import import_crnt4sbml
from sympy.matrices.utilities import dotprodsimp

crnt4sbml = import_crnt4sbml()
stage, objs = {}, []

def timed(name, fn, *a, **kw):
    t = time.time()
    out = fn(*a, **kw)
    stage[name] = time.time() - t
    print(f"{name:38s}{stage[name]:9.1f} s", flush=True)
    return out

with dotprodsimp(False):                   # as the tool runs crnt4sbml (sympy behaviour it was written for)
    net = timed("read network", crnt4sbml.CRNT, CRNT_SBML)
    cg = timed("reaction graph", net.get_c_graph)
    print(f"  species {len(cg.get_species())}, complexes {len(cg.get_complexes())}, "
          f"reactions {len(cg.get_reactions())}")
    try:
        mca = timed("mass-conservation setup", net.get_mass_conservation_approach)
        bounds, cbounds = timed("optimisation bounds", mca.get_optimization_bounds)
        params, objs = timed(f"optimisation ({ITERATIONS} iterations)", mca.run_optimization,
                             bounds=bounds, concentration_bounds=cbounds,
                             iterations=ITERATIONS, seed=0)
    except Exception as e:
        print("mass-conservation approach not applicable:", type(e).__name__, str(e)[:200])

hits = [o for o in objs if o <= 1e-10]
print(f"\n{len(hits)} candidate set(s) with objective ~ 0 (several steady states)")

## 3. The tool, exactly as the agent calls it

In [ ]:
t = time.time()
r = json.loads(crnt_bistability(MODEL, iterations=ITERATIONS))
print(f"crnt_bistability: {time.time() - t:.1f} s")
print("error:", r.get("error"))
print("conclusion:", r.get("conclusion"))
print("result fields:", list(r))

## 4. Optional: the same question through the agent (needs the API key)

In [ ]:
from dynalyst.agents.unified_agent import run

t = time.time()
rec = run(f"Use CRNT on {MODEL} and give me candidate bistability parameter sets, if there are any.")
print(f"agent: {time.time() - t:.1f} s")
print(rec.final_answer[:2000])

## If the network is the problem: options

| Option | Valid? | Notes |
|---|---|---|
| Fewer iterations | yes | Only shortens the optimisation stage. "No candidates" from a short search is inconclusive. |
| Search near the model's own values first | yes | The tool already does this before crnt4sbml's generic ranges. |
| Analyse one module (e.g. one layer of a cascade) as its own mass-action network | partly | A module's multistationarity is a lead, not a proof for the full network: coupling can create or remove it. |
| QSSA / timescale reduction first | no | Reduction gives non-mass-action rate laws; crnt4sbml reads every reaction as mass action, so the CRNT answer would be about a different network. |
| Merge enzyme-binding steps | no | Changes the network; not equivalent. |
| crnt4sbml's MPI-parallel optimisation | yes | Splits the optimisation over cores (see crnt4sbml docs); does not speed up the setup stage. |
| Skip CRNT: bifurcation or LHS scan directly | yes | AUTO-07p looks for folds numerically; finds bistability but proves nothing when none is found. |